1- **Create Table on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Available_Visits"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "Visit_Key",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Time_Table_Key",
        "STRING"
    ),

    bigquery.SchemaField(
        "Dr_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "Speciality",
        "STRING"
    ),

    bigquery.SchemaField(
        "Dr_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scheduled_Day",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scheduled_Date",
        "DATE"
    ),

    bigquery.SchemaField(
        "Scheduled_Start_Time",
        "TIME"
    ),

    bigquery.SchemaField(
        "Scheduled_End_Time",
        "TIME"
    ),

    bigquery.SchemaField(
        "Shift_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Capacity",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Remaining_Capacity",
        "INTEGER"
    )

]

# =========================================
# CREATE TABLE
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# PARTITIONING
# =========================================

table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="Scheduled_Date"
)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "Dr_Code",
    "Scheduled_Date"
]

# =========================================
# CREATE TABLE
# =========================================

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("Partitioned By: Scheduled_Date")
print("Clustered By: Dr_Code, Scheduled_Date")
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Available_Visits
Partitioned By: Scheduled_Date
Clustered By: Dr_Code, Scheduled_Date


2- **Create Table on Postgres**

In [3]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# Connect to PostgreSQL
conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# Create Sequence for Visit_Key
cursor.execute("CREATE SEQUENCE IF NOT EXISTS visit_no_seq;")

# Create Normalized Table
create_table_query = """
CREATE TABLE IF NOT EXISTS Available_Visits (
    Visit_Key TEXT PRIMARY KEY DEFAULT (
        'VK' || LPAD(nextval('visit_no_seq')::TEXT, 6, '0')
    ),
    Time_Table_Key TEXT NOT NULL,
    Scheduled_Date DATE NOT NULL,
    Capacity INTEGER CHECK (Capacity >= 0),
    Remaining_Capacity INTEGER CHECK (Remaining_Capacity >= 0)
);
"""
cursor.execute(create_table_query)

# Indexes
cursor.execute("CREATE INDEX IF NOT EXISTS idx_avail_tt ON Available_Visits(Time_Table_Key);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_avail_visit_date ON Available_Visits(Scheduled_Date);")

#Uniqueness constraint to avoid duplicate slots (Time_Table_Key + Scheduled_Date)
cursor.execute("""

CREATE UNIQUE INDEX IF NOT EXISTS uq_visit_slot
ON Available_Visits (
    Time_Table_Key,
    Scheduled_Date
);

""")

conn.commit()
cursor.close()
conn.close()
print("PostgreSQL table 'Available_Visits' created (or already exists).")

PostgreSQL table 'Available_Visits' created (or already exists).


**Sync BigQuery to Postgres**

In [2]:
import pandas as pd
from psycopg2.extras import execute_values
from google.cloud import bigquery
from google.oauth2 import service_account
import sys
import psycopg2
sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# Authentication
SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
client = bigquery.Client(credentials=credentials, project="depihealthnux")

# Read from BigQuery
query = """
SELECT
    Visit_Key,
    Time_Table_Key,
    Scheduled_Date,
    Capacity,
    Remaining_Capacity
FROM `depihealthnux.Depihealthnux_Main.Available_Visits`
ORDER BY Visit_Key;
"""
df = client.query(query).to_dataframe()

# Insert into PostgreSQL
conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()
cursor.execute("DELETE FROM Available_Visits;")

if not df.empty:
    execute_values(cursor,
        """
        INSERT INTO Available_Visits (
            Visit_Key, 
            Time_Table_Key, 
            Scheduled_Date, 
            Capacity, 
            Remaining_Capacity
        ) VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# Reset sequence
cursor.execute("""
SELECT setval(
    'visit_no_seq',
    COALESCE(
        (SELECT MAX(CAST(REPLACE(Visit_Key,'VK','') AS INTEGER)) FROM Available_Visits),
        1
    ),
    true
);
""")

conn.commit()
print(f"Inserted {len(df)} rows into PostgreSQL 'Available_Visits'.")

# Verify
cursor.execute("SELECT * FROM Available_Visits ORDER BY Visit_Key LIMIT 3;")
print("\nFirst 3 Rows From PostgreSQL:")
for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Inserted 0 rows into PostgreSQL 'Available_Visits'.

First 3 Rows From PostgreSQL:


**Sync Postgres to BigQuery**

In [4]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

# =========================================
# READ FROM POSTGRES
# =========================================

query = """

SELECT

    v.Visit_Key,

    v.Time_Table_Key,

    tt.Dr_Code,

    d.Speciality,

    d.Dr_Name,

    tt.Day_of_Week AS Scheduled_Day,

    v.Scheduled_Date,

    tt.Scheduled_Start_Time,

    tt.Scheduled_End_Time,

    CONCAT(
        d.Dr_Name,
        ' - ',
        d.Speciality,
        ' - ',
        tt.Day_of_Week,
        ' - ',
        TO_CHAR(v.Scheduled_Date,'YYYY-MM-DD'),
        ' - ',
        TO_CHAR(tt.Scheduled_Start_Time,'HH12:MI AM')
    ) AS Shift_Name,

    v.Capacity,

    v.Remaining_Capacity

FROM Available_Visits v

LEFT JOIN Dr_Time_Table tt
    ON v.Time_Table_Key = tt.Time_Table_Key

LEFT JOIN Dr_List d
    ON tt.Dr_Code = d.Dr_Code

ORDER BY v.Visit_Key

"""

df = pd.read_sql(query, conn)

conn.close()

# =========================================
# FIX DATA TYPES FOR BIGQUERY
# =========================================

# DATE Partition Column
df["scheduled_date"] = pd.to_datetime(
    df["scheduled_date"]
).dt.date

# TIME Columns
df["scheduled_start_time"] = pd.to_datetime(
    df["scheduled_start_time"].astype(str),
    format="%H:%M:%S"
).dt.time

df["scheduled_end_time"] = pd.to_datetime(
    df["scheduled_end_time"].astype(str),
    format="%H:%M:%S"
).dt.time

# =========================================
# VERIFY DATA TYPES
# =========================================

print("=================================")
print("Data Types")
print("=================================")
print(df.dtypes)

print("\n=================================")
print("First 3 Rows From PostgreSQL")
print("=================================")
print(df.head(3))

# =========================================
# LOAD TO BIGQUERY
# =========================================
print(df.columns.tolist())
print(df.dtypes)
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)
if df.empty:

    print("=================================")
    print("No Available Visits Found")
    print("BigQuery Sync Skipped")
    print("=================================")

else:

    job = client.load_table_from_dataframe(
        df,
        "depihealthnux.Depihealthnux_Main.Available_Visits",
        job_config=job_config
    )

    job.result()

    print(f"Uploaded {len(df)} rows")

# =========================================
# VERIFY BIGQUERY DATA
# =========================================

verify_query = """

SELECT
    Visit_Key,
    Time_Table_Key,
    Dr_Code,
    Speciality,
    Dr_Name,
    Scheduled_Day,
    Scheduled_Date,
    Scheduled_Start_Time,
    Scheduled_End_Time,
    Shift_Name,
    Capacity,
    Remaining_Capacity
FROM `depihealthnux.Depihealthnux_Main.Available_Visits`
ORDER BY Visit_Key
LIMIT 3

"""

verify_df = client.query(
    verify_query
).to_dataframe()

print("\n=================================")
print("First 3 Rows From BigQuery")
print("=================================")
print(verify_df)

print("\n=================================")
print("SYNC COMPLETED SUCCESSFULLY")
print("=================================")

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_11268\3704839285.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data Types
visit_key               object
time_table_key          object
dr_code                 object
speciality              object
dr_name                 object
scheduled_day           object
scheduled_date          object
scheduled_start_time    object
scheduled_end_time      object
shift_name              object
capacity                 int64
remaining_capacity       int64
dtype: object

First 3 Rows From PostgreSQL
  visit_key time_table_key dr_code speciality    dr_name scheduled_day  \
0  VK000001          TT001   Dr001      Chest  Dr. Wafaa        Monday   
1  VK000002          TT001   Dr001      Chest  Dr. Wafaa        Monday   
2  VK000003          TT001   Dr001      Chest  Dr. Wafaa        Monday   

  scheduled_date scheduled_start_time scheduled_end_time  \
0     2026-06-01             18:00:00           21:00:00   
1     2026-06-08             18:00:00           21:00:00   
2     2026-06-15             18:00:00           21:00:00   

                                   